# 📝 Build & Sign Cardano Transaction — Step-by-Step Guide

Notebook này build, ký và chuẩn bị giao dịch Cardano theo workflow `build-and-sign-tx.md`.

| Bước | Mô tả |
|------|-------|
| 0 | Cấu hình môi trường & tham số giao dịch |
| 1A | Build tx với automatic fee (cần protocol-params) |
| 1B | Build raw tx (offline, tính fee thủ công) |
| 2 | Ký giao dịch (`tx.signed`) |
| 3 | Trích xuất Transaction ID |
| 4 | Submit giao dịch |
| 5 | Multi-signature (partial signing) |

> **⚠️ Lưu ý:** Thay `<txhash>`, `<index>`, địa chỉ, số ADA bằng giá trị thật của bạn trước khi chạy.

## 0. Cấu hình môi trường & tham số giao dịch

Điền các thông tin giao dịch của bạn vào cell dưới đây:

In [ ]:
import subprocess
import os
import json
from pathlib import Path

# ── Đường dẫn cardano-cli.exe ──────────────────────────────
CARDANO_CLI = r"d:\Blockchain\tooldev\cardano-cli-win64\cardano-cli-11.0.0.0-win64\cardano-cli.exe"

# ── Thư mục làm việc (chứa payment.skey, payment.vkey, payment.addr) ──
WORK_DIR = Path(r"d:\Blockchain\tooldev\cardano-cli-win64\cardano-cli-11.0.0.0-win64\wallet-keys")
os.chdir(WORK_DIR)

# ════════════════════════════════════════════════════════════
#  THAM SỐ GIAO DỊCH — ĐIỀN GIÁ TRỊ THẬT CỦA BẠN
# ════════════════════════════════════════════════════════════

# UTxO input (lấy từ block explorer, ví dụ Cardanoscan)
TX_IN_HASH    = "a1b2c3d4e5f6...."   # ← thay bằng tx hash thật
TX_IN_INDEX   = 0                      # ← index của UTxO

# Người nhận
RECIPIENT_ADDR = "addr1q...."          # ← địa chỉ người nhận
SEND_AMOUNT_ADA = 10                   # ← số ADA gửi

# Địa chỉ nhận tiền thừa (change) — thường là payment.addr của bạn
CHANGE_ADDR_FILE = WORK_DIR / "payment.addr"

# File key
PAYMENT_SKEY = WORK_DIR / "payment.skey"
PAYMENT_VKEY = WORK_DIR / "payment.vkey"

# File protocol parameters (cho Option B — download từ node/API)
PROTOCOL_PARAMS_FILE = WORK_DIR / "protocol-parameters.json"

# ── Đơn vị: 1 ADA = 1,000,000 lovelace ─────────────────────
LOVELACE_PER_ADA = 1_000_000
SEND_AMOUNT_LOVELACE = SEND_AMOUNT_ADA * LOVELACE_PER_ADA

# ── Kiểm tra ───────────────────────────────────────────────
print("=" * 60)
print("  📋 THÔNG TIN GIAO DỊCH")
print("=" * 60)
print(f"  cardano-cli     : {CARDANO_CLI}")
print(f"  Working dir     : {WORK_DIR}")
print(f"  Tx-in hash      : {TX_IN_HASH}")
print(f"  Tx-in index     : {TX_IN_INDEX}")
print(f"  Recipient addr  : {RECIPIENT_ADDR}")
print(f"  Send amount     : {SEND_AMOUNT_ADA} ADA ({SEND_AMOUNT_LOVELACE:,} lovelace)")
print(f"  Change addr     : {CHANGE_ADDR_FILE}")
print(f"  payment.skey    : {'✅ exists' if PAYMENT_SKEY.exists() else '❌ MISSING'}")
print(f"  payment.vkey    : {'✅ exists' if PAYMENT_VKEY.exists() else '❌ MISSING'}")
print(f"  protocol-params : {'✅ exists' if PROTOCOL_PARAMS_FILE.exists() else '⚠️  MISSING (cần cho Option B)'}")
print("=" * 60)

---
## Option A — Build với automatic fee

Cách này CLI tự tính fee dựa trên protocol parameters. Cần file `protocol-parameters.json` (download từ node hoặc API).

```bash
cardano-cli conway transaction build \
  --conway-era \
  --tx-in <txhash>#<index> \
  --tx-out <recipient-address>+<amount-in-lovelace> \
  --change-address <your-payment-address> \
  --out-file tx.raw
```

> CLI tự động: tính fee, trừ fee từ change, tạo output change về địa chỉ của bạn.

In [ ]:
tx_raw = WORK_DIR / "tx.raw"
tx_in = f"{TX_IN_HASH}#{TX_IN_INDEX}"
tx_out = f"{RECIPIENT_ADDR}+{SEND_AMOUNT_LOVELACE}"
change_addr = CHANGE_ADDR_FILE.read_text().strip()

cmd = [
    CARDANO_CLI, "conway", "transaction", "build",
    "--conway-era",
    "--tx-in", tx_in,
    "--tx-out", tx_out,
    "--change-address", change_addr,
    "--out-file", str(tx_raw),
]

# Nếu có protocol-params, thêm vào (giúp CLI tính fee chính xác)
if PROTOCOL_PARAMS_FILE.exists():
    cmd += ["--protocol-params-file", str(PROTOCOL_PARAMS_FILE)]
    print("  → Sử dụng protocol-params.json để tính fee")
else:
    print("  ⚠️  Không có protocol-params.json — build có thể thất bại")

print(f"  → Tx-in  : {tx_in}")
print(f"  → Tx-out : {tx_out}")
print(f"  → Change : {change_addr}")
print()

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Lỗi:")
    print(result.stderr)
else:
    print("✅ Đã build tx.raw (automatic fee)")
    print(result.stdout.strip())
    print(f"   File: {tx_raw}")

---
## Option B — Build raw (offline, tính fee thủ công)

Cách này hoàn toàn offline. Bạn phải tự tính fee và tự tạo output change (tiền thừa).

### Bước B1: Tính minimum fee

```bash
cardano-cli conway transaction calculate-min-fee \
  --tx-body-file tx.raw \
  --tx-in-count 1 \
  --tx-out-count 2 \
  --witness-count 1 \
  --protocol-parameters-file protocol-parameters.json
```

In [ ]:
# ⚠️ Chạy cell này SAU khi đã build-raw ở cell dưới (B2) lần đầu,
# hoặc tạo file tx.raw tạm trước. Thông thường:
#   1. Build-raw với fee=0 → tx.raw
#   2. calculate-min-fee → ra fee
#   3. Build-raw lại với fee chính xác

if not PROTOCOL_PARAMS_FILE.exists():
    print("❌ Cần protocol-parameters.json để tính fee!")
    print("   Download từ node hoặc API (Blockfrost/Koios)")
else:
    result = subprocess.run(
        [CARDANO_CLI, "conway", "transaction", "calculate-min-fee",
         "--tx-body-file", str(tx_raw),
         "--tx-in-count", "1",
         "--tx-out-count", "2",
         "--witness-count", "1",
         "--protocol-parameters-file", str(PROTOCOL_PARAMS_FILE)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ Lỗi:", result.stderr)
    else:
        output = result.stdout.strip()
        print("✅ Fee calculation:")
        print(f"   {output}")
        # Parse fee (format: "12345 lovelace")
        parts = output.split()
        if parts:
            fee_lovelace = int(parts[0])
            fee_ada = fee_lovelace / LOVELACE_PER_ADA
            print(f"   = {fee_lovelace:,} lovelace ({fee_ada:.6f} ADA)")

### Bước B2: Build raw transaction với fee chính xác

```bash
cardano-cli conway transaction build-raw \
  --conway-era \
  --tx-in <txhash>#<index> \
  --tx-out <recipient-address>+<amount> \
  --tx-out <change-address>+<change-amount> \
  --fee <fee-in-lovelace> \
  --out-file tx.raw
```

> **Công thức change:** `change = UTxO_amount - send_amount - fee`

In [ ]:
# ════════════════════════════════════════════════════════════
#  ĐIỀN FEE & CHANGE AMOUNT (từ bước B1)
# ════════════════════════════════════════════════════════════

# Fee (lovelace) — lấy từ calculate-min-fee ở B1
FEE_LOVELACE = 170000          # ← thay bằng fee thật từ B1

# UTxO amount (tổng lovelace trong UTxO input)
UTXO_AMOUNT_LOVELACE = 50_000_000   # ← thay bằng số dư UTxO thật

# Tính change = UTxO - send - fee
CHANGE_AMOUNT_LOVELACE = UTXO_AMOUNT_LOVELACE - SEND_AMOUNT_LOVELACE - FEE_LOVELACE

print(f"  UTxO amount  : {UTXO_AMOUNT_LOVELACE:,} lovelace ({UTXO_AMOUNT_LOVELACE/LOVELACE_PER_ADA} ADA)")
print(f"  Send amount  : {SEND_AMOUNT_LOVELACE:,} lovelace ({SEND_AMOUNT_ADA} ADA)")
print(f"  Fee          : {FEE_LOVELACE:,} lovelace ({FEE_LOVELACE/LOVELACE_PER_ADA:.6f} ADA)")
print(f"  Change       : {CHANGE_AMOUNT_LOVELACE:,} lovelace ({CHANGE_AMOUNT_LOVELACE/LOVELACE_PER_ADA:.6f} ADA)")
print()

assert CHANGE_AMOUNT_LOVELACE > 0, "❌ Change âm — kiểm tra lại UTxO amount!"

tx_in = f"{TX_IN_HASH}#{TX_IN_INDEX}"
tx_out_send = f"{RECIPIENT_ADDR}+{SEND_AMOUNT_LOVELACE}"
tx_out_change = f"{change_addr}+{CHANGE_AMOUNT_LOVELACE}"

result = subprocess.run(
    [CARDANO_CLI, "conway", "transaction", "build-raw",
     "--conway-era",
     "--tx-in", tx_in,
     "--tx-out", tx_out_send,
     "--tx-out", tx_out_change,
     "--fee", str(FEE_LOVELACE),
     "--out-file", str(tx_raw)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã build tx.raw (raw, manual fee)")
    print(f"   File: {tx_raw}")

---
## Bước 2 — Ký giao dịch (`tx.signed`)

Dùng `payment.skey` để ký transaction body.

```bash
cardano-cli conway transaction sign \
  --tx-body-file tx.raw \
  --signing-key-file payment.skey \
  --out-file tx.signed
```

In [ ]:
tx_signed = WORK_DIR / "tx.signed"

result = subprocess.run(
    [CARDANO_CLI, "conway", "transaction", "sign",
     "--tx-body-file", str(tx_raw),
     "--signing-key-file", str(PAYMENT_SKEY),
     "--out-file", str(tx_signed)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã ký giao dịch → tx.signed")
    print(f"   File: {tx_signed}")
    print(f"   Size: {tx_signed.stat().st_size} bytes")
    # Hiển thị metadata (ẩn cborHex)
    with open(tx_signed, "r") as f:
        data = json.load(f)
    print(f"   type        : {data.get('type')}")
    print(f"   description : {data.get('description')}")
    print(f"   cborHex     : [HIDDEN — giao dịch đã ký]")

---
## Bước 3 — Trích xuất Transaction ID

Tx ID là hash duy nhất của giao dịch — dùng để tra cứu trên block explorer.

```bash
cardano-cli conway transaction txid --tx-file tx.signed
```

In [ ]:
result = subprocess.run(
    [CARDANO_CLI, "conway", "transaction", "txid",
     "--tx-file", str(tx_signed)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    tx_id = result.stdout.strip()
    print("✅ Transaction ID:")
    print()
    print(f"   🔖 {tx_id}")
    print()
    print(f"   🔍 Tra cứu: https://cardanoscan.io/transaction/{tx_id}")
    print(f"   🔍 Cexplorer: https://cexplorer.io/tx/{tx_id}")

---
## Bước 4 — Submit giao dịch

### Cách 1: Submit qua node (cần node socket)

In [ ]:
# ⚠️ Cần Cardano node đang chạy. Điền socket path thật.
# Trên Windows dùng CARDANO_NODE_SOCKET_PATH env var.

NODE_SOCKET_PATH = r"\\.\pipe\cardano-node"   # ← thay bằng socket path thật

SUBMIT = False  # ← Đổi thành True để submit thật

if SUBMIT:
    env = os.environ.copy()
    env["CARDANO_NODE_SOCKET_PATH"] = NODE_SOCKET_PATH

    result = subprocess.run(
        [CARDANO_CLI, "conway", "transaction", "submit",
         "--tx-file", str(tx_signed)],
        capture_output=True, text=True,
        env=env
    )

    if result.returncode != 0:
        print("❌ Lỗi:", result.stderr)
    else:
        print("✅ Giao dịch đã được submit!")
        print(result.stdout.strip())
else:
    print("  ⏸️  Submit đang tắt (SUBMIT=False)")
    print("  Đổi SUBMIT=True và điền NODE_SOCKET_PATH để submit.")

### Cách 2: Submit qua dịch vụ bên ngoài

Nếu không chạy node, submit `tx.signed` qua:

| Dịch vụ | URL |
|----------|-----|
| CardanoScan | https://cardanoscan.io/tx-submit |
| Blockfrost API | https://docs.blockfrost.io/#tag/Transactions/paths/~1tx~1submit/post |
| Koios API | https://api.koios.rest/#post-/tx_id |
| Wallet (Lace, Daedalus, Eternl) | Import tx.signed vào ví → submit |

In [ ]:
# Ví dụ submit qua Blockfrost API (cần API key)

BLOCKFROST_PROJECT_ID = ""   # ← điền Blockfrost project_id
SUBMIT_VIA_API = False        # ← Đổi True để submit

if SUBMIT_VIA_API and BLOCKFROST_PROJECT_ID:
    import requests

    # Đọc tx.signed dưới dạng CBOR bytes
    with open(tx_signed, "r") as f:
        tx_data = json.load(f)
    # Blockfrost cần raw CBOR — dùng cardano-cli convert hoặc gửi cborHex
    cbor_hex = tx_data.get("cborHex", "")
    tx_bytes = bytes.fromhex(cbor_hex)

    url = "https://cardano-mainnet.blockfrost.io/api/v0/tx/submit"
    headers = {"project_id": BLOCKFROST_PROJECT_ID}
    resp = requests.post(url, headers=headers, data=tx_bytes)

    if resp.status_code == 202:
        print("✅ Submitted! Tx ID:", resp.text)
    else:
        print(f"❌ Lỗi {resp.status_code}:", resp.text)
else:
    print("  ⏸️  Submit qua API đang tắt")
    print("  Điền BLOCKFROST_PROJECT_ID và SUBMIT_VIA_API=True để submit.")

---
## Bước 5 — Multi-signature (Partial Signing)

Dùng khi giao dịch cần nhiều chữ ký (multi-sig, cold/hot wallet).

### 5a. Tạo witness trên máy ký

In [ ]:
tx_witness = WORK_DIR / "tx.witness"

# Dùng signing key để tạo witness (không cần tx.raw full — chỉ cần body)
result = subprocess.run(
    [CARDANO_CLI, "conway", "transaction", "witness",
     "--tx-body-file", str(tx_raw),
     "--signing-key-file", str(PAYMENT_SKEY),
     "--out-file", str(tx_witness)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã tạo witness → tx.witness")
    print(f"   File: {tx_witness}")

### 5b. Assemble — ghép witness vào tx body

In [ ]:
tx_assembled = WORK_DIR / "tx.signed"

result = subprocess.run(
    [CARDANO_CLI, "conway", "transaction", "assemble",
     "--tx-body-file", str(tx_raw),
     "--witness-file", str(tx_witness),
     "--out-file", str(tx_assembled)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Lỗi:", result.stderr)
else:
    print("✅ Đã assemble → tx.signed")
    print(f"   File: {tx_assembled}")
    print(f"   Size: {tx_assembled.stat().st_size} bytes")

---
## 📋 Tổng kết — Kiểm tra file đã tạo

In [ ]:
expected_files = [
    ("tx.raw",      "📄 Transaction body (chưa ký)"),
    ("tx.signed",   "✍️  Transaction đã ký — sẵn sàng submit"),
    ("tx.witness",  "🔑 Witness (cho multi-sig)"),
]

print("=" * 60)
print("  📦 TRANSACTION FILES — Tổng kết")
print("=" * 60)
print()

for filename, desc in expected_files:
    filepath = WORK_DIR / filename
    if filepath.exists():
        size = filepath.stat().st_size
        print(f"  ✅ {filename:<14} {size:>8} bytes  {desc}")
    else:
        print(f"  ➖ {filename:<14} {'—':>10}      {desc} (chưa tạo)")

print()
print("=" * 60)
print("  💡 UNIT CONVERSION")
print("=" * 60)
print("  1 ADA = 1,000,000 lovelace")
print(f"  {SEND_AMOUNT_ADA} ADA = {SEND_AMOUNT_LOVELACE:,} lovelace")
print("=" * 60)